# EEGConformer vs PCA Evaluation Notebook

This notebook evaluates the existing EEGConformer ONNX model against the PCA baseline in Neuro-Fabric.

**Steps:**
1. Load and preprocess BCI-IV-2a data (using the same pipeline as Neuro-Fabric).
2. Load the pre-trained EEGConformer ONNX model.
3. Compute EEGConformer embeddings for the embeddings via ONNX Runtime.
4. Compute PCA embeddings (fit on training data, transform on test).
5. Compare the two methods on classification performance, embedding quality, inference latency, and more.


In [ ]:
# Setup environment and repository
import os
import json
import numpy as np
from pathlib import Path

REPO_URL = "https://github.com/ouadaououhoussemdroid/neuro-fabric-core.git"
REPO_DIR = Path("/content/neuro-fabric-core")

if not REPO_DIR.exists():
    print("📥 Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR} > /dev/null 2>&1
else:
    print("📁 Repository exists - pulling latest...")
    !cd {REPO_DIR} && git pull > /dev/null 2>&1

    os.chdir(str(REPO_DIR))
print(f"📍 Working directory: {os.getcwd()}")

# Load configuration
CONFIG_PATH = REPO_DIR / "training" / "configs" / "eegconformer-bciiv2a.yaml"
with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

# Extract parameters
CHANNELS = config["signal"]["channels"]
SAMPLE_RATE = config["signal"]["sample_rate_hz"]
WINDOW_SAMPLES = config["signal"]["window_samples"]
N_CLASSES = config["model"]["n_classes"]
SUBJECTS = config["dataset"]["subjects"][:2]  # FIRST 2 SUBJECTS ONLY
HOLDOUT = set([SUBJECTS[-1]]) if len(SUBJECTS) > 1 else set()  # HOLD OUT LAST

print(f"📊 Using {len(SUBJECTS)} subjects for training, {len(HOLDOUT)} for testing")
print(f"   Subjects: {SUBJECTS}")
print(f"   Holdout: {list(HOLDOUT)}")


In [ ]:
# Install dependencies
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# List of packages
packages = [
    "numpy",
    "scipy",
    "scikit-learn",
    "onnxruntime",
    "moabb",
    "pandas",
    "matplotlib",
    "seaborn",
    "nbformat"
]

for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        install(package)



In [ ]:
# Load and preprocess data
import moabb
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np
import os

moabb.set_log_level("WARNING")

sig = {
    "channels": CHANNELS,
    "sample_rate_hz": SAMPLE_RATE,
    "bandpass_hz": config["signal"]["bandpass_hz"],
    "tmin_s": config["signal"]["tmin_s"],
    "tmax_s": config["signal"]["tmax_s"],
    "window_samples": WINDOW_SAMPLES
}
paradigm = MotorImagery(
    fmin=sig["bandpass_hz"][0],
    fmax=sig["bandpass_hz"][1],
    tmin=sig["tmin_s"],
    tmax=sig["tmax_s"],
    resample=sig["sample_rate_hz"],
    n_classes=N_CLASSES
)

dataset = BNCI2014_001()
print(f"📥 Loading data for subjects {SUBJECTS}...")
X, labels, metadata = paradigm.get_data(dataset=dataset, subjects=SUBJECTS)
X = np.asarray(X, dtype=np.float32)

# Handle string labels (MOABB quirk)
if labels.dtype.kind in 'UOS':  # If labels are strings/unicode
    print(f"🔤 Detected string labels: {np.unique(labels)}")
    # Map string labels to integers for BCI-IV-2a
    label_mapping = {
        'left_hand': 0,
        'right_hand': 1,
        'feet': 2,
        'tongue': 3
    }
    # Convert labels using list comprehension
    try:
        labels = np.array([label_mapping.get(str(label).strip().lower()) for label in labels])
    except Exception as e:
        raise ValueError(f"Label mapping failed: {e}") from e
    # Check for unknown labels
    if np.any(labels == None):
        unknown_labels = set(labels) - set(label_mapping.values())
        raise ValueError(f"Unknown labels found: {unknown_labels}") from e

# Now convert to int64
labels = np.asarray(labels, dtype=np.int64)

# Contract enforcement
target_T = WINDOW_SAMPLES
if X.shape[-1] != target_T:
    if X.shape[-1] > target_T:
        X = X[..., :target_T]
    else:
        pad_width = ((0, 0), (0, 0), (0, target_T - X.shape[-1]))
        X = np.pad(X, pad_width, mode="constant")

# Per-trial z-score per channel
def _zscore(x: np.ndarray) -> np.ndarray:
    # x: [N, C, T] -> per-trial per-channel
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True) + 1e-6
    return ((x - mean) / std).astype(np.float32)

X = _zscore(X)
y = np.asarray(labels, dtype=np.int64)
subjects_arr = np.asarray(metadata["subject"], dtype=np.int64)

# Split into train and test based on holdout subject
train_mask = np.isin(subjects_arr, list(set(subjects) - holdout))
test_mask = np.isin(subjects_arr, holdout)

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
subjects_train = subjects_arr[train_mask]
subjects_test = subjects_arr[test_mask]

print(f"✅ DATA LOADING COMPLETE:")
print(f"   Training samples: {X_train.shape[0]}")
print(f"   Test samples: {X_test.shape[0]}")
print(f"   Data shape: {X_train.shape[1:]} (channels, samples)")

# FREE RAW DATA IMMEDIATELY
del X, labels, metadata
import gc
gc.collect()



In [ ]:
# Load the ONNX model
import onnxruntime as ort

MODEL_PATH = REPO_DIR / "public" / "models" / "eegconformer.onnx"
print(f"📥 Loading ONNX model from: {MODEL_PATH}")

session = ort.InferenceSession(str(MODEL_PATH))

# Get input and output details
input_name = session.get_inputs()[0].name
output_names = [output.name for output in session.get_outputs()]

print(f"📥 Input name: {input_name}")
print(f"📥 Output names: {output_names}")

# Verify the input shape
input_shape = session.get_inputs()[0].shape
print(f"📥 Expected input shape: {input_shape}")

# We expect the input to be [batch, channels, samples] (as per the export script)
# Let's run a dummy input to see the output shapes and names.
dummy_input = np.random.randn(1, CHANNELS, WINDOW_SAMPLES).astype(np.float32)
dummy_output = session.run(None, {input_name: dummy_input})
print(f"🧪 Dummy output shapes: {[o.shape for o in dummy_output]}")
print(f"🧪 Output names corresponding to shapes: {list(zip(output_names, [o.shape for o in dummy_output]))}")

# Determine which output is the embedding and which is the logits.
# From the export script, the first output is embedding, second is logits.
# But we can also check the shapes: embedding should be [1, 32], logits [1, 4]
embedding_idx = 0
logits_idx = 1
if len(dummy_output) >= 2:
    if dummy_output[0].shape[1] == config.get("embedding_dim", 32):
        embedding_idx = 0
        logits_idx = 1
    elif dummy_output[1].shape[1] == config.get("embedding_dim", 32):
        embedding_idx = 1
        logits_idx = 0
    else:
        # Default to first output if neither matches expected embedding dim
        embedding_idx = 0
        logits_idx = 1 if len(dummy_output) > 1 else None

print(f"🎯 Embedding output index: {embedding_idx}")
if logits_idx is not None:
    print(f"🎯 Logits output index: {logits_idx}")



In [ ]:
# Function to run EEGConformer inference on a batch of data
def run_eegconformer_onnx(X_batch):
    """Run EEGConformer inference on a batch of data."""
    # Determine if we need to add a dimension based on model's expected input shape
    input_shape = session.get_inputs()[0].shape
    # If the model expects a 4D input [batch, 1, channels, samples], we need to add a dimension
    if len(input_shape) == 4 and input_shape[1] == 1:
        # Add a dimension at position 1 (after batch)
        X_batch = np.expand_dims(X_batch, axis=1)
    # Run the model
    outputs = session.run(None, {input_name: X_batch})
    # Return the embedding (using the embedding_idx we determined earlier)
    return outputs[embedding_idx]

# Compute EEGConformer embeddings for train and test data
print("⏳ Computing EEGConformer embeddings...")
import time
import gc
start_time = time.time()

# Process in batches to avoid memory spikes
BATCH_SIZE = 64

def compute_embeddings_in_batches(X_data, desc=""):
    embeddings = []
    total_samples = len(X_data)
    
    for i in range(0, total_samples, BATCH_SIZE):
        # Get batch
        batch_end = min(i + BATCH_SIZE, total_samples)
        batch = X_data[i:batch_end].copy()
        
        # Process batch
        batch_emb = run_eegconformer_onnx(batch)
        embeddings.append(batch_emb)
        
        # IMMEDIATE CLEANUP
        del batch, batch_emb
        gc.collect()
        
        # Progress reporting
        if i % (BATCH_SIZE * 5) == 0 or batch_end == total_samples:
            print(f"  {desc}: {batch_end}/{total_samples} samples ({100*batch_end/total_samples:.1f}%)")
    
    # Combine results
    if embeddings:
        result = np.vstack(embeddings)
    else:
        result = np.array([]).reshape(0, X_data.shape[1] if len(X_data.shape) > 1 else 0)
    
    return result

# Process train and test sets
X_train_emb = compute_embeddings_in_batches(X_train, "Training")
X_test_emb = compute_embeddings_in_batches(X_test, "Test")

eegconformer_time = time.time() - start_time

print(f"✅ EEGConformer embeddings computed in {eegconformer_time:.2f}s")
print(f"   Training embeddings shape: {X_train_emb.shape}")
print(f"   Test embeddings shape: {X_test_emb.shape}")

# FREE TRAIN/TEST DATA IMMEDIATELY
del X_train, X_test
gc.collect()



In [ ]:
# Compute PCA embeddings (fit on training data, transform on test)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

print("⏳ Computing PCA embeddings...")
pca_start = time.time()

# Flatten the data for PCA (from [N, C, T] to [N, C*T])
n_train, c, t = X_train_emb.shape
n_test = X_test_emb.shape[0]
X_train_flat = X_train_emb.reshape(n_train, -1)
X_test_flat = X_test_emb.reshape(n_test, -1)

# Standardize features (important for PCA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

# Apply PCA
n_components = X_train_emb.shape[1]  # same as EEGConformer embedding dimension
pca = PCA(n_components=n_components)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

pca_time = time.time() - pca_start

print(f"✅ PCA embeddings computed in {pca_time:.2f}s")
print(f"   Training PCA embeddings shape: {X_train_pca.shape}")
print(f"   Test PCA embeddings shape: {X_test_pca.shape}")
print(f"   Explained variance ratio (first 5 components): {pca.explained_variance_ratio_[:5]}")
print(f"   Total explained variance ({n_components} components): {np.sum(pca.explained_variance_ratio_):.3f}")

# FREE SCALED DATA AND SCALER IMMEDIATELY
del X_train_scaled, X_test_scaled, scaler
gc.collect()



In [ ]:
# Train classifiers and evaluate performance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import time

print("⏳ Training classifiers and evaluating performance...")

# EEGConformer embeddings
clf_ec = LogisticRegression(max_iter=1000, random_state=42)
clf_ec.fit(X_train_emb, y_train)
y_pred_ec = clf_ec.predict(X_test_emb)
acc_ec = accuracy_score(y_test, y_pred_ec)
print(f"🎯 EEGConformer + Logistic Regression accuracy: {acc_ec:.4f}")

# PCA embeddings
clf_pca = LogisticRegression(max_iter=1000, random_state=42)
clf_pca.fit(X_train_pca, y_train)
y_pred_pa = clf_pca.predict(X_test_pca)
acc_pa = accuracy_score(y_test, y_pred_pa)
print(f"🎯 PCA + Logistic Regression accuracy: {acc_pa:.4f}")

# Compare the two approaches
print("\n📊 COMPARISON:")
print(f"   EEGConformer accuracy: {acc_ec:.4f}")
print(f"   PCA accuracy:          {acc_pa:.4f}")
print(f"   Difference (EEGConformer - PCA): {acc_ec - acc_pa:.4f}")

# Inference latency (per sample) - we already have the total time for embeddings
# We can compute the average time per sample for EEGConformer
eegconformer_latency_per_sample = eegconformer_time / len(X_train_emb)  # training time per sample
pca_latency_per_sample = pca_time / len(X_train_emb)  # PCA fitting time per sample (note: PCA fitting is done once)

print(f"⏱️  EEGConformer latency per sample (training): {eegconformer_latency_per_sample*1000:.2f} ms")
print(f"⏱️  PCA latency per sample (fitting): {pca_latency_per_sample*1000:.2f} ms")

# Store results for later use
eval_results = {
    "eegconformer_accuracy": float(acc_ec),
    "pca_accuracy": float(acc_pa),
    "eegconformer_latency_per_sample_ms": float(eegconformer_latency_per_sample * 1000),
    "pca_latency_per_sample_ms": float(pca_latency_per_sample * 1000),
    "training_samples": int(len(X_train_emb)),
    "test_samples": int(len(X_test_emb)),
    "embedding_dimension": int(X_train_emb.shape[1]),
}



In [ ]:
# Save results to a JSON file
import json
from pathlib import Path

# Define the output directory
output_dir = REPO_DIR / "training" / "artefacts" / "eegconformer-bciiv2a-v1"
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "colab_evaluation_report.json"

# Prepare the final report
report = {
    "repository": str(REPO_DIR),
    "configuration": {
        "channels": CHANNELS,
        "sample_rate_hz": SAMPLE_RATE,
        "window_samples": WINDOW_SAMPLES,
        "embedding_dimension": X_train_emb.shape[1],
        "n_classes": N_CLASSES,
    },
    "results": eval_results,
}

# Write the JSON file
with open(output_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"💾 Evaluation report saved to: {output_file}")

print("🎉 EVALUATION COMPLETE!")
